# Autonomous Code Vulnerability Detection - Demo Notebook

This notebook walks through the full pipeline:
1. Build the dataset
2. Run the static rule engine
3. Run the ML classifier (offline TF-IDF fallback by default)
4. Fuse them into an explainable report
5. Evaluate all three strategies (rules / model / hybrid)

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from data import build_dataset
from explanation.rules import RuleEngine
from models.detector import CodeVulnerabilityDetector

## 1. Dataset

In [ ]:
ds = build_dataset('builtin', augment=True)
print('records:', len(ds))
print('balance:', ds.class_balance())

## 2. Static rule engine

In [ ]:
engine = RuleEngine()
code = 'import os\n\ndef run(x):\n    os.system("ls " + x)\n'
for f in engine.scan(code, 'python'):
    print(f.line, f.cwe_id, f.cwe.name, '->', f.message)

## 3-4. Full analysis (model + rules + explanation)

In [ ]:
det = CodeVulnerabilityDetector(backend='heuristic')
print('backend:', det.backend)
report = det.analyse(code, 'python')
print(json.dumps(report.to_dict(), indent=2))

## 5. Ablation: rules vs model vs hybrid

In [ ]:
from models.heuristic import HeuristicClassifier
from evaluation import classification_metrics, format_metrics

train, val, test = ds.split()
clf = HeuristicClassifier().fit(train.texts(), train.labels())
model_scores = clf.predict_proba_batch(test.texts())
rule_scores = [max([f.confidence for f in engine.scan(c, l)] or [0.0])
               for c, l in zip(test.texts(), [r['language'] for r in test.records])]
hybrid = [max(m, r) for m, r in zip(model_scores, rule_scores)]
labels = test.labels()

print('rules-only :', format_metrics(classification_metrics(labels, rule_scores)))
print('model-only :', format_metrics(classification_metrics(labels, model_scores)))
print('hybrid     :', format_metrics(classification_metrics(labels, hybrid)))